# 03. 가설 검정 및 최종 변수 선정 (다영 담당)

대상: **가설2**(단시간 내 고액(500불↑) 반복결제 / burst spending), **가설3**(업종별 동적 사기율 / Data Drift 적응형 피처), 그리고 다영님이 추가로 검증한 **거리(distance_km_2)/속도(speed_2)**.

`02_EDA_전체파이프라인`에서 이미 수행한 통계검정(t-test, 카이제곱)을 그대로 가져와 재확인하고, 여기에 **다른 검정 방법(비모수 검정), 효과크기, 신뢰구간, 검정력(신뢰도), 다른 위험요인과의 교차분석(상호작용 검정), 최종 변수 선정**을 추가한다. raw 컬럼과 다영님이 직접 만든 파생변수만 사용한다.

## 0. 데이터 로드 (팀 병합 파일에서 담당 컬럼만 선택)

`fraudTrain_feature_engineering.csv`에서 raw 컬럼(존재하는 것만)과 다영님이 만든 파생변수 4개만 가져온다. 다른 팀원이 만든 파생변수(`night_transaction`, `customer_mean_amt`, `count_30min` 등)는 사용하지 않는다.

*참고*: 이 파일에는 다영님이 원래 부르던 `distance_km`/`speed_kmh`라는 이름이 그대로 없고 대신 `distance_km_2`, `speed_2`가 있다 — 팀 병합 과정에서 다른 팀원이 만든 동명 피처와 이름이 겹쳐서 붙은 접미사로 보인다(충돌 없이 살아남은 `speed_kmh`라는 이름은 평균 165.3/최댓값 780,623으로 값 자체가 다영님 로직과 다름 — 다른 팀원 것이라 사용하지 않는다). `distance_km_2`(평균 76.1, n=1,090,393)와 `speed_2`(평균 102.4, 최댓값 15,820)는 `02_EDA_전체파이프라인`에서 확인했던 다영님 본인의 값과 정확히 일치해 다영님 파생변수로 확인했다. **파일의 컬럼명은 그대로 두고(rename 없이), 이후 코드에서도 `distance_km_2`, `speed_2`라는 이름을 그대로 사용한다.**

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pointbiserialr

usecols = [
    'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt',
    'lat', 'long', 'dob', 'trans_num', 'merch_lat', 'merch_long', 'is_fraud',
    'recent_24h_high_amt_count', 'category_recent_fraud_rate',
    'distance_km_2', 'speed_2',
]
df = pd.read_csv('fraudTrain_feature_engineering.csv', usecols=usecols)

print(df.shape)
df.head()

(1296675, 16)


,trans_date_trans_time,cc_num,merchant,category,amt,lat,long,dob,trans_num,merch_lat,merch_long,is_fraud,recent_24h_high_amt_count,category_recent_fraud_rate,distance_km_2,speed_2
0,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,42.1808,-112.2620,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,43.150704,-112.154481,0,0.0,0.005789,108.206083,NaN
1,2019-01-01 01:23:26,180014262259255,fraud_Stark-Batz,entertainment,5.83,31.5710,-86.2743,1958-06-26,14eb123e2dc9745d0ca4c87a5469ffe4,31.798954,-86.213243,0,0.0,0.000000,25.997384,NaN
2,2019-01-01 01:28:22,3592325941359225,fraud_Grimes LLC,entertainment,7.06,37.2876,-77.2950,1935-08-15,d27fd7a3300953b0cb67d38e65c2f344,37.426207,-76.502881,0,0.0,0.000000,71.688075,357.290052
3,2019-01-01 02:02:04,3514897282719543,"fraud_Nicolas, Hills and McGlynn",entertainment,145.75,42.9580,-77.3083,1952-10-13,debedd2588067ec99d29d26c343d90b6,42.446162,-78.048599,0,0.0,0.000000,83.057990,NaN
4,2019-01-01 02:10:12,4740713119940984,fraud_Brown-Greenholt,entertainment,5.27,41.1901,-74.0436,1962-10-16,9f55bd65f64193f1b1a46a99c93e7844,41.788775,-74.471154,0,0.0,0.000000,75.496531,NaN


In [14]:
# 교차분석에 쓸 구간(bucket) — 02_EDA의 교차 히트맵과 동일한 경계
df['amt_count_group'] = pd.cut(
    df['recent_24h_high_amt_count'], bins=[-0.1, 0, 1, 2, 3, 100], labels=['0', '1', '2', '3', '4+']
)
df['speed_group'] = pd.cut(
    df['speed_2'], bins=[-0.1, 10, 30, 60, 100, 300, 1_000_000],
    labels=['0-10', '10-30', '30-60', '60-100', '100-300', '300+']
)

## 가설 2 검정 — 단시간 내 고액(500불↑) 반복결제(burst spending)

**H1**: 최근 24시간 내 500불 이상 결제 반복횟수(`recent_24h_high_amt_count`)가 많을수록 이상거래일 가능성이 높다.
**H0**: `recent_24h_high_amt_count`는 `is_fraud`와 관련이 없다.

In [15]:
# 02_EDA에서 이미 수행한 t-test 재확인 + 분포 특성을 고려한 추가 검정(Mann-Whitney U)
normal_group = df.loc[df['is_fraud'] == 0, 'recent_24h_high_amt_count']
fraud_group = df.loc[df['is_fraud'] == 1, 'recent_24h_high_amt_count']

t_stat, p_ttest = stats.ttest_ind(normal_group, fraud_group, equal_var=False)
print(f'[t-test] 정상 평균={normal_group.mean():.3f}, 사기 평균={fraud_group.mean():.3f}, t={t_stat:.2f}, p={p_ttest:.2e}')

u_stat, p_mw = stats.mannwhitneyu(normal_group, fraud_group, alternative='less')
print(f'[Mann-Whitney U, 단측] U={u_stat:.4e}, p={p_mw:.2e}')

[t-test] 정상 평균=0.039, 사기 평균=1.693, t=-91.02, p=0.00e+00
[Mann-Whitney U, 단측] U=1.5084e+09, p=0.00e+00


*참고*: `recent_24h_high_amt_count`는 대부분 거래에서 0이고 일부만 1 이상인 극단적으로 치우친(zero-inflated) 분포라, t-test가 가정하는 "두 그룹이 정규분포에 가깝다"는 전제가 잘 안 맞을 수 있다. 분포 모양에 의존하지 않는 비모수 검정인 Mann-Whitney U를 교차검증으로 추가했다.

In [16]:
corr2, p_corr2 = pointbiserialr(df['is_fraud'], df['recent_24h_high_amt_count'])
n = len(df)
z2 = np.arctanh(corr2)
se2 = 1 / np.sqrt(n - 3)
ci2 = (np.tanh(z2 - 1.96 * se2), np.tanh(z2 + 1.96 * se2))

pooled_std = np.sqrt(
    ((normal_group.var() * (len(normal_group) - 1)) + (fraud_group.var() * (len(fraud_group) - 1)))
    / (len(normal_group) + len(fraud_group) - 2)
)
cohens_d = (fraud_group.mean() - normal_group.mean()) / pooled_std

u_two, _ = stats.mannwhitneyu(normal_group, fraud_group, alternative='two-sided')
cles2 = 1 - u_two / (len(normal_group) * len(fraud_group))  # 임의의 사기거래 값이 임의의 정상거래 값보다 클 확률

print(f'point-biserial 상관계수: {corr2:.4f}  (95% CI: {ci2[0]:.4f} ~ {ci2[1]:.4f})')
print(f"Cohen's d: {cohens_d:.2f}")
print(f'공통언어효과크기(CLES): {cles2:.1%}')

point-biserial 상관계수: 0.4579  (95% CI: 0.4565 ~ 0.4592)
Cohen's d: 6.79
공통언어효과크기(CLES): 84.4%


해석: point-biserial 상관계수 0.458(95% CI 0.456~0.459)로 지금까지 확인한 어떤 변수보다 강한 연관성을 보인다. Cohen's d는 6.79로 관례적 기준(0.8=큼)을 훨씬 넘는데, 이 변수가 대부분 0에 몰린 극단적 비대칭 분포라 d 값이 부풀려졌을 가능성이 있어 그대로 "효과가 이만큼 크다"고 해석하기보다는 참고용으로만 본다. 대신 분포 모양에 영향받지 않는 공통언어효과크기(CLES)를 보면, 무작위로 뽑은 사기거래 하나가 무작위로 뽑은 정상거래보다 `recent_24h_high_amt_count`가 더 큰 경우가 84.4%로, 실질적으로도 매우 강력하게 구분되는 변수임을 확인했다. t-test(p≈0)와 Mann-Whitney U(p≈0) 모두 같은 결론을 가리켜, 검정 방법을 바꿔도 결론이 달라지지 않는다는 것도 확인했다.

In [17]:
# 신뢰도(검정력, statistical power) 확인 — "진짜 차이가 있다면, 지금 표본 크기로 그걸 놓치지 않고 잡아낼 확률"
from statsmodels.stats.power import TTestIndPower

power2 = TTestIndPower().power(
    effect_size=cohens_d, nobs1=len(fraud_group), ratio=len(normal_group) / len(fraud_group),
    alpha=0.05, alternative='larger'
)
print(f'검정력(power): {power2:.4f}  (표본: 정상 {len(normal_group):,}건, 사기 {len(fraud_group):,}건)')

검정력(power): 1.0000  (표본: 정상 1,289,169건, 사기 7,506건)


해석: 검정력 100%(1.0000)로, 지금 관측된 정도의 차이가 실제로 존재한다면 이 표본 크기로는 거의 확실히 통계적으로 유의미하게 잡아낼 수 있다는 뜻이다. 표본이 130만 건에 달해 "거래 건수가 부족해서 놓쳤을 가능성"은 사실상 없다고 봐도 된다.

## 가설 3 검정 — 업종별 동적 사기율 (Data Drift 적응형 피처)

**H1**: 가맹점의 고정된 업종 정보보다, 업종별 최근 7일 사기비율(`category_recent_fraud_rate`)이 이상거래와 더 밀접하게 관련된다.
**H0**: 업종(또는 `category_recent_fraud_rate`)은 `is_fraud`와 관련이 없다.

In [18]:
# 02_EDA에서 이미 수행한 카이제곱(범주형 category vs is_fraud) 재확인
contingency = pd.crosstab(df['category'], df['is_fraud'])
chi2_stat, p_chi, dof, expected = stats.chi2_contingency(contingency)
print(f'[카이제곱] chi2={chi2_stat:.2f}, p={p_chi:.2e}, 자유도={dof}')

# category_recent_fraud_rate(연속형 동적 피처) 자체에 대한 추가 검정
normal3 = df.loc[df['is_fraud'] == 0, 'category_recent_fraud_rate']
fraud3 = df.loc[df['is_fraud'] == 1, 'category_recent_fraud_rate']
u3, p_mw3 = stats.mannwhitneyu(normal3, fraud3, alternative='less')
print(f'[Mann-Whitney U, 단측] U={u3:.4e}, p={p_mw3:.2e}')

[카이제곱] chi2=6486.00, p=0.00e+00, 자유도=13
[Mann-Whitney U, 단측] U=2.5901e+09, p=0.00e+00


In [19]:
corr3, p_corr3 = pointbiserialr(df['is_fraud'], df['category_recent_fraud_rate'])
z3 = np.arctanh(corr3)
ci3 = (np.tanh(z3 - 1.96 * se2), np.tanh(z3 + 1.96 * se2))

u3_two, _ = stats.mannwhitneyu(normal3, fraud3, alternative='two-sided')
cles3 = 1 - u3_two / (len(normal3) * len(fraud3))

print(f'point-biserial 상관계수: {corr3:.4f}  (95% CI: {ci3[0]:.4f} ~ {ci3[1]:.4f})')
print(f'공통언어효과크기(CLES): {cles3:.1%}')

point-biserial 상관계수: 0.0711  (95% CI: 0.0694 ~ 0.0728)
공통언어효과크기(CLES): 73.2%


해석: 업종(category)과 사기여부는 카이제곱 검정에서 확실히 관련이 있다(p≈0). 다만 `category_recent_fraud_rate` 자체의 상관계수는 0.071(95% CI 0.069~0.073)로 약한 편이다. 대신 CLES로 보면 무작위 사기거래의 `category_recent_fraud_rate`가 무작위 정상거래보다 큰 경우가 73.2%로, 상관계수만 보고 "약하다"고 단정하기보다는 방향성 있는 실질적 차이가 존재한다고 볼 수 있다. 다만 이 수치는 `recent_24h_high_amt_count`(84.4%)보다는 명백히 약해서, "가장 강력한 변수는 아니지만 보조 신호로서 유의미하다"는 원래 결론과 일치한다.

In [20]:
# 신뢰도(검정력) 확인 — category_recent_fraud_rate(연속형)와 category(범주형) 둘 다
from statsmodels.stats.power import GofChisquarePower

pooled_std3 = np.sqrt(
    ((normal3.var() * (len(normal3) - 1)) + (fraud3.var() * (len(fraud3) - 1)))
    / (len(normal3) + len(fraud3) - 2)
)
cohens_d3 = (fraud3.mean() - normal3.mean()) / pooled_std3
power3 = TTestIndPower().power(
    effect_size=cohens_d3, nobs1=len(fraud3), ratio=len(normal3) / len(fraud3),
    alpha=0.05, alternative='larger'
)

cohens_w = np.sqrt(chi2_stat / n)
power_chi = GofChisquarePower().power(effect_size=cohens_w, nobs=n, alpha=0.05, n_bins=dof + 1)

print(f'category_recent_fraud_rate 검정력: {power3:.4f}  (Cohen\'s d={cohens_d3:.2f})')
print(f"category 카이제곱 검정력: {power_chi:.4f}  (Cohen's w={cohens_w:.4f})")

category_recent_fraud_rate 검정력: 1.0000  (Cohen's d=0.94)
category 카이제곱 검정력: 1.0000  (Cohen's w=0.0707)


해석: 두 검정 모두 검정력이 100%로, 표본 크기가 부족해서 실제 효과를 놓쳤을 가능성은 없다. 다만 검정력이 100%라는 건 "차이를 감지할 능력이 충분하다"는 뜻이지 "효과가 크다"는 뜻은 아니다 — 실제로 `category_recent_fraud_rate`의 상관계수(0.071)는 여전히 약한 편이며, 검정력과 효과크기는 서로 다른 질문에 답한다는 걸 유의해야 한다.

## 추가 검증 — 거리(distance_km_2)/속도(speed_2)

`02_EDA`에서 상관계수만 봤던(distance 0.0004, speed 0.02) 두 변수에 실제 통계적 유의성 검정(피어슨 상관계수 유의성 검정)을 추가해, "상관계수가 작다"는 게 "통계적으로도 무의미하다"는 뜻인지 아니면 "유의미하지만 효과가 작다"는 뜻인지 구분한다.

In [21]:
dist_valid = df.dropna(subset=['distance_km_2'])
r_dist, p_dist = stats.pearsonr(dist_valid['distance_km_2'], dist_valid['is_fraud'])
print(f'distance_km_2: r={r_dist:.4f}, p={p_dist:.3f}, n={len(dist_valid):,}')

speed_valid = df.dropna(subset=['speed_2'])
r_speed, p_speed = stats.pearsonr(speed_valid['speed_2'], speed_valid['is_fraud'])
print(f'speed_2: r={r_speed:.4f}, p={p_speed:.2e}, n={len(speed_valid):,}')

distance_km_2: r=0.0004, p=0.693, n=1,090,393
speed_2: r=0.0179, p=5.56e-78, n=1,089,410


해석: `distance_km_2`는 상관계수 0.0004에 p=0.693으로, 130만 건에 가까운 표본으로도 0과 통계적으로 구분되지 않는다 — 이건 "효과가 작다"가 아니라 "진짜 관련이 없다"는 뜻이라 최종 변수에서 제외해도 안전하다. 반면 `speed_2`는 상관계수 0.018로 여전히 작지만 p=5.6e-78로 통계적으로는 명백히 유의미하다 — 표본이 워낙 커서(약 109만 건) 아주 작은 관련성도 우연이 아니라고 확인되는 경우로, 통계적 유의성과 실무적 중요도가 다를 수 있다는 걸 다시 보여준다.

## 다른 위험요인과 교차분석 (상호작용 검정)

`02_EDA`의 교차 히트맵에서 `speed_2`와 `recent_24h_high_amt_count`를 같이 보면 특정 구간(고속+반복 4회 이상)에서 사기율이 86.67%까지 치솟는 걸 봤는데, 이게 그래프상 우연히 튄 건지 통계적으로 실재하는 상호작용인지 로지스틱 회귀로 formalize한다.

In [22]:
import statsmodels.formula.api as smf

# 1) 연속형 곱으로 상호작용 검정 (속도와 반복횟수가 부드럽게 비례해서 상승하는 관계인지)
d_offline = df.dropna(subset=['speed_2']).copy()
m_linear = smf.logit('is_fraud ~ recent_24h_high_amt_count * speed_2', data=d_offline).fit(disp=0)
print(m_linear.summary().tables[1])

                                        coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------
Intercept                            -6.3104      0.022   -289.450      0.000      -6.353      -6.268
recent_24h_high_amt_count             2.4290      0.020    121.681      0.000       2.390       2.468
speed_2                               0.0002   2.38e-05      6.695      0.000       0.000       0.000
recent_24h_high_amt_count:speed_2  1.609e-05   2.32e-05      0.695      0.487   -2.93e-05    6.15e-05


In [23]:
# 2) 히트맵에서 쓴 구간(bucket)별 상호작용 검정 -> 주효과만 있는 모델 vs 구간별 상호작용까지 포함한 모델을
#    우도비 검정(likelihood ratio test)으로 비교 (연속형 곱과 달리 비선형/문턱값 형태의 상호작용도 잡아냄)
from scipy.stats import chi2 as chi2_dist

d_offline_g = d_offline.dropna(subset=['amt_count_group', 'speed_group'])
m_main = smf.logit('is_fraud ~ C(amt_count_group) + C(speed_group)', data=d_offline_g).fit(disp=0)
m_full = smf.logit('is_fraud ~ C(amt_count_group) * C(speed_group)', data=d_offline_g).fit(disp=0)

lr_stat = 2 * (m_full.llf - m_main.llf)
lr_df = m_full.df_model - m_main.df_model
p_lr = chi2_dist.sf(lr_stat, lr_df)
print(f'우도비검정: LR={lr_stat:.2f}, df={lr_df:.0f}, p={p_lr:.2e}')

우도비검정: LR=122.93, df=20, p=8.14e-17


해석: 흥미롭게도 두 검정이 다른 결론을 준다. `recent_24h_high_amt_count`×`speed_2`를 단순 곱(연속형)으로 검정하면 상호작용 항이 유의하지 않다(p=0.487) — 즉 "속도가 빠를수록, 반복횟수가 많을수록 부드럽게 비례해서 위험이 커진다"는 형태의 상호작용은 아니다. 반면 히트맵에서 쓴 구간(bucket)별로 상호작용을 검정(우도비 검정)하면 p≈8.1e-17로 뚜렷하게 유의하다 — 즉 특정 구간 조합(고속+반복 다수)에서만 위험이 비선형적으로 튀는 상호작용이 통계적으로도 실재한다는 뜻이다. 결론적으로 히트맵에서 본 86.67% 패턴은 우연이 아니라 진짜 신호이지만, 모델에 넣을 때는 단순 곱셈 상호작용 피처보다 구간화(bucketing)나 트리 기반 모델처럼 비선형 상호작용을 잡을 수 있는 방식이 더 적합하다는 것을 시사한다.

In [24]:
# 가설2 변수 x 가설3 변수의 상호작용도 함께 확인 (두 담당 가설이 서로 보강되는지)
m_interact23 = smf.logit(
    'is_fraud ~ recent_24h_high_amt_count * category_recent_fraud_rate', data=df
).fit(disp=0)
print(m_interact23.summary().tables[1])

                                                           coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------
Intercept                                               -6.6631      0.026   -257.905      0.000      -6.714      -6.612
recent_24h_high_amt_count                                2.3184      0.025     93.423      0.000       2.270       2.367
category_recent_fraud_rate                              72.0734      1.767     40.798      0.000      68.611      75.536
recent_24h_high_amt_count:category_recent_fraud_rate    27.3234      2.112     12.936      0.000      23.183      31.463


해석: 가설2 변수와 가설3 변수를 곱한 상호작용 항은 계수 27.32, p≈0으로 뚜렷하게 유의하다 — `speed_2`와 달리 이 조합은 단순 곱(연속형) 형태로도 상호작용이 확인된다. 즉 "평소 그 업종에 사기가 많았던 시기에, 이 카드가 동시에 고액을 반복결제했다면" 위험이 두 변수를 따로 봤을 때의 합보다 더 크게 뛴다는 뜻으로, 두 담당 가설(2, 3)의 변수를 곱한 새 피처(`recent_24h_high_amt_count × category_recent_fraud_rate`)를 모델링 단계 후보로 추가할 근거가 된다.

## 최종 변수 선정

| 변수 | 관련 가설 | 효과크기 | 검정 결과 | 최종 판단 |
|---|---|---|---|---|
| `recent_24h_high_amt_count` | 가설2 | r=0.458, CLES=84.4% | t-test·Mann-Whitney 모두 p≈0 | **채택 (최우선 변수)** |
| `category_recent_fraud_rate` | 가설3 | r=0.071, CLES=73.2% | Mann-Whitney p≈0 | **채택** (단독 효과는 약함) |
| `category`(범주형) | 가설3 | 카이제곱 6486.00 | p≈0 | **채택** (동적 피처와 병행 여부는 모델링 단계에서 결정) |
| `distance_km_2` | 추가검증 | r=0.0004 | **p=0.693 (불유의)** | **제외** |
| `speed_2` | 추가검증 | r=0.018 | p≈0 (통계적 유의, 효과 미미) | **보류** — 단독 채택 근거 약함, `recent_24h_high_amt_count`와의 연속형 상호작용도 비유의(p=0.487) |
| `recent_24h_high_amt_count`×`category_recent_fraud_rate` (신규 상호작용 피처) | 가설2×3 | 상호작용계수 27.32 | p≈0 | **모델링 단계 후보로 추가** |

**요약**: 가설2(`recent_24h_high_amt_count`)가 가장 강력한 단일 변수임을 t-test·Mann-Whitney U·상관계수·CLES 네 가지 방법으로 재확인했다. 가설3(`category_recent_fraud_rate`)은 단독 효과는 약하지만 방향성 있는 유의미한 신호이며, 가설2 변수와의 상호작용에서 더 강하게 나타난다. `distance_km_2`는 통계적으로도 완전히 무관함이 확인되어 제외를 확정하고, `speed_2`는 기존에 서술했던 "반복횟수와 결합 시 강함"이라는 결론이 연속형 상호작용 검정에서는 재현되지 않아(구간별 검정에서는 유의) 모델링 단계에서 신중히 다뤄야 한다는 점을 새롭게 확인했다.